# Chapter 3: Enabling actions: Tool use

In [2]:
import json
import os
from tavily import TavilyClient
from dotenv import load_dotenv
from litellm import completion
load_dotenv()

True

In [3]:
import inspect

### Listing 3.1: Calculator Tool Definition Schema

In [2]:
calculator_tool_definition = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Perform basic arithmetic operations.",
        "parameters": {
            "type": "object",
            "properties": {
                "operator": {
                    "type": "string",
                    "description": "Arithmetic operation to perform",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "first_number": {
                    "type": "number",
                    "description": "First number for the calculation"
                },
                "second_number": {
                    "type": "number",
                    "description": "Second number for the calculation"
                }
            },
            "required": ["operator", "first_number", "second_number"]
        }
    }
}          

### Listing 3.2: Implementation of Calculator Function

In [3]:
def calculator(operator: str, first_number: float, second_number: float):
    if operator == "add":
        return first_number + second_number
    elif operator == "subtract":
        return first_number - second_number
    elif operator == "multiply":
        return first_number * second_number
    elif operator == "divide":
        if second_number == 0:
            raise ValueError("Cannot divide by zero")
        return first_number / second_number
    else:
        raise ValueError(f"Unsupported operator: {operator}")

### Listing 3.3: Tool-calling execution example

In [ ]:
tools = [calculator_tool_definition]

response_without_tool = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "What is the capital of South Korea?"}],
    tools=tools
)

print(response_without_tool.choices[0].message.content)
print(response_without_tool.choices[0].message.tool_calls)

response_with_tool = completion(
        model='gpt-5-mini',
        messages=[{"role": "user", "content": "What is 1234 x 5678?"}],
        tools=tools
)
print(response_with_tool.choices[0].message.content)
print(response_with_tool.choices[0].message.tool_calls)

### Experiment: Smartbeings `check_minibar` example

The function checks whether items are available in the guest's room minibar, before suggesting or ordering anything.

In [4]:
minibar_menu = ["rum", "pineapple_juice", "coconut_cream", "cola", "water"]
check_minibar_tool_definition = {
    "type": "function",
    "function": {
        "name": "check_minibar",
        "description": "Check whether items are available in the guest's room minibar, before suggesting or ordering anything.",
        "parameters": {
            "type": "object",
            "properties": {
                "items": {
                    "type": "array",
                    "description": "Minibar items to check, e.g. the ingredients for a requested drink",
                    "items": {"type": "string", "enum": minibar_menu},
                    "minItems": 1
                },
            },
            "required": ["items"],
        },
    },
}     

In [5]:
tools = [calculator_tool_definition, check_minibar_tool_definition]

response_without_tool = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "What is the capital of South Korea?"}],
    tools=tools
)

print(response_without_tool.choices[0].message.content)
print(response_without_tool.choices[0].message.tool_calls)

response_with_tool = completion(
        model='gpt-5-mini',
        messages=[{"role": "user", "content": "What is 1234 x 5678?"}],
        tools=tools
)

print(response_with_tool.choices[0].message.content)
print(response_with_tool.choices[0].message.tool_calls)

response_minibar = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "Can I make a piña colada with what's in my minibar?"}],
    tools=tools
)

print(response_minibar.choices[0].message.content)
print(response_minibar.choices[0].message.tool_calls)

The capital of South Korea is Seoul (서울특별시). 

Note: some government ministries and agencies were moved to Sejong City for administrative functions, but Seoul remains the country's official capital.
None
None
[ChatCompletionMessageToolCall(function=Function(arguments='{"operator":"multiply","first_number":1234,"second_number":5678}', name='calculator'), id='call_QlxyGL3naTbbE0A7jwLVDts7', type='function')]
None
[ChatCompletionMessageToolCall(function=Function(arguments='{"items":["rum","pineapple_juice","coconut_cream"]}', name='check_minibar'), id='call_r1CzNNBjPOYJMYMP6zukQyRO', type='function')]


### Listing 3.4: Extracting and executing the tool from the response

In [6]:
ai_message = response_with_tool.choices[0].message

if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

if function_name == "calculator":
    result = calculator(**function_args)

In [7]:
print(type(tool_call.function.arguments), type(function_args), result)

<class 'str'> <class 'dict'> 7006652


### Listing 3.5: Feeding tool results back to the LLM

In [8]:
messages=[{"role": "user", "content": "What is 1234 x 5678?"}]
messages.append({"role": "assistant", "content": ai_message.content, "tool_calls": ai_message.tool_calls})

if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

        if function_name == "calculator":
            result = calculator(**function_args)

            messages.append({"role": "tool", "content": str(result), "tool_call_id": tool_call.id})

final_response = completion(
    model="gpt-5-mini",
    messages=messages
)
print("Messages:", messages)
print("Final_answer:", final_response.choices[0].message.content)

Messages: [{'role': 'user', 'content': 'What is 1234 x 5678?'}, {'role': 'assistant', 'content': None, 'tool_calls': [ChatCompletionMessageToolCall(function=Function(arguments='{"operator":"multiply","first_number":1234,"second_number":5678}', name='calculator'), id='call_QlxyGL3naTbbE0A7jwLVDts7', type='function')]}, {'role': 'tool', 'content': '7006652', 'tool_call_id': 'call_QlxyGL3naTbbE0A7jwLVDts7'}]
Final_answer: 1234 × 5678 = 7,006,652


In [9]:
print("Final answer:", final_response.choices[0].message.content)

Final answer: 1234 × 5678 = 7,006,652


### Listing 3.12: Basic Web Search Function Using Tavily

In [6]:
tavily_client = TavilyClient(os.getenv("TAVILY_API_KEY"))

def search_web(query: str, max_results: int=2) -> list:
    response = tavily_client.search(query, max_results=max_results)
    return response.get("results")                        

### Listing 3.13: Testing the Web Search Function

In [ ]:
search_web("Kipchoge's marathon world record")

### Listing 3.15: Web Search Function with Error Handling

In [7]:
def search_web(query: str, max_results: int = 5, topic: str = "general", time_range: str | None = None) -> list | str:
    """Search the web for the given query"""
    try:
        response = tavily_client.search(query, max_results=max_results, topic=topic, time_range=time_range)
        return response.get("results")

    except Exception as e:
        return f"Error: Search Failed - {e}"

### Listing 3.16: Extracting Tool Metadata Using `inspect`

In [12]:
import inspect

def example_tool(input_1: str, input_2: int=1):
    """docstring for example_tool"""
    return

print(f"function name: {example_tool.__name__}")
print(f"function docstring: {example_tool.__doc__}")
print(f"function signature: {inspect.signature(example_tool)}")

function name: example_tool
function docstring: docstring for example_tool
function signature: (input_1: str, input_2: int = 1)


### Listing 3.17: Generating input schema from python functions

In [4]:
def function_to_input_schema(func) -> dict:
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }

    try:
        signature = inspect.signature(func)
    except ValueError as e:
        raise ValueError(f"Failed to get signature for function: {func.__name__}: {str(e)}")

    parameters = {}
    for param in signature.parameters.values():
        param_type = type_map.get(param.annotation, "string") #unknown types fall back to "string" (e.g. list[str] -> wrong!) 
        parameters[param.name] = {"type": param_type}

    required = [param.name for param in signature.parameters.values() if param.default == inspect._empty]
    return {"type": "object", "properties": parameters, "required": required}

### Listing 3.18: Converting functions to tool definitions

In [8]:
def format_tool_definition(name: str, description:str, parameters:dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }

def function_to_tool_definition(func) -> dict:
    return format_tool_definition(
        func.__name__,
        func.__doc__ or "",
        function_to_input_schema(func)
    )

function_to_tool_definition(search_web)

{'type': 'function',
 'function': {'name': 'search_web',
  'description': 'Search the web for the given query',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string'},
    'max_results': {'type': 'integer'},
    'topic': {'type': 'string'},
    'time_range': {'type': 'string'}},
   'required': ['query']}}}

### Listing 3.19: Tool-Execution Utility Function

In [15]:
def tool_execution(tool_box, tool_call):
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    tool_result = tool_box[function_name](**function_args)
    return tool_result

### Execution 3.20: Simple Agent Loop for Tool Execution

In [18]:
def simple_agent_loop(system_prompt, question, max_steps: int = 5) -> str:
    tools = [search_web]
    tool_box = {tool.__name__ : tool for tool in tools}
    tool_definitions = [function_to_tool_definition(tool) for tool in tools]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]

    for step in range(max_steps):
        print(f"step {step}: calling LLM ({len(messages)} messages)")
        response = completion(
            model="gpt-5-mini",
            messages=messages,
            tools=tool_definitions
        )

        assistant_message = response.choices[0].message

        if assistant_message.tool_calls:
            messages.append(assistant_message)
            for tool_call in assistant_message.tool_calls:
                tool_result = tool_execution(tool_box, tool_call)
                messages.append({
                    "role": "tool",
                    "content": str(tool_result),
                    "tool_call_id": tool_call.id
                })

        else:
            return assistant_message.content

    return "Stopped: too many tool calls"

### Listing 3.21: Testing the Agent Loop

In [19]:
system_prompt = """You are a helpful assistant.
Use the search tool when you need current information."""

result = simple_agent_loop(system_prompt, "Who won the 2025 Nobel Prize in Physics?")

print(result)

step 0: calling LLM (2 messages)
step 1: calling LLM (4 messages)
step 2: calling LLM (6 messages)
step 3: calling LLM (8 messages)
The 2025 Nobel Prize in Physics was awarded jointly to John Clarke, Michel H. Devoret and John M. Martinis — "for the discovery of macroscopic quantum mechanical tunnelling and energy quantisation in an electric circuit." (Source: NobelPrize.org)


In [1]:
import sys; print(sys.executable)

/Users/sharonjasminerasheed/code/ai-agent-from-scratch/.venv/bin/python


In [9]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


### Listing 3.22 Connecting to an MCP Server

In [11]:
server_params = StdioServerParameters(
    command="npx",
    args=["-y", "tavily-mcp@latest"],
    env={
        "TAVILY_API_KEY": os.getenv("TAVILY_API_KEY"),
    }
)

async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        #List available tools
        tools_result = await session.list_tools()
        print("Available tools:")
        for tool in tools_result.tools:
            print(f"  - {tool.name}: {tool.description[:60]}...")

        result = await session.call_tool(
            "tavily_search",
            arguments={"query": "2025 Nobel Physics"}
        )
        print("Search result:")
        print(result.content)

Available tools:
  - tavily_search: Search the web for current information on any topic. Use for...
  - tavily_extract: Extract content from URLs. Returns raw page content in markd...
  - tavily_crawl: Crawl a website starting from a URL. Extracts content from p...
  - tavily_map: Map a website's structure. Returns a list of URLs found star...
  - tavily_research: Perform comprehensive research on a given topic or question....
Search result:
[TextContent(type='text', text='Detailed Results:\n\nTitle: John Clarke, Michel Devoret and John Martinis win the 2025 Nobel Prize for Physics – Physics World\nID: 04c7bc-00\nURL: https://physicsworld.com/a/john-clarke-michel-devoret-and-john-martinis-win-the-2025-nobel-prize-for-physics\nContent: John Clarke, Michel Devoret and John Martinis share the 2025 Nobel Prize for Physics “for the discovery of macroscopic quantum mechanical tunnelling and energy quantization in an electric circuit”. \n\nThe award includes a SEK 11m prize ($1.2m), which is 

### Listing 3.23 Converting MCP tools to OpenAI format

In [15]:
def mcp_tools_to_openai_format(mcp_tools) -> list[dict]:
    """Convert MCP tool definitions to OpenAI tool format."""

    return [
        format_tool_definition(
            name=tool.name,
            description=tool.description,
            parameters=tool.input_schema,
        )
        for tool in mcp_tools.tools
    ]

### Listing 3.24: Retrieving tools in OpenAI format

In [16]:
async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        #List available tools
        tool_result = await session.list_tools()
        openai_format_tools = mcp_tools_to_openai_format(tool_result)
        for tool in openai_format_tools:
            print(tool)

{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month',

### Listing 3.25: Custom Tavily MCP Server Implementation

Look at the `tavily_mcp_server.py` file

### Listing 3.26 Connecting to the custom MCP Server

In [19]:
server_params = StdioServerParameters(
    command="uv",
    args=["run", "tavily_mcp_server.py"],
    env={"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}
)

async with stdio_client(server_params) as (read_stream, write_stream):
    async with ClientSession(read_stream, write_stream) as session:
        await session.initialize()

        # List available tools
        tools_result = await session.list_tools()
        print("Available tools:")
        for tool in tools_result.tools:
            print(f"  - {tool.name}: {tool.description}")

Available tools:
  - search_web: 
Search the Web using Tavily API.

Args:
    query: Search query string
    max_results: Maximum number of results to return (default: 5)

Returns:
    Search results as formatted string



In [18]:
import os; print(os.getcwd())

/Users/sharonjasminerasheed/code/ai-agent-from-scratch/chapter_03
